All supplementary tables:

Table S1:

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import pandas as pd


# ============================================================
# Locate repository
# ============================================================

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "src" / "scrna_benchmark").is_dir():
            return p

    raise RuntimeError(
        "Run this notebook from somewhere inside the repository."
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(
        0,
        str(REPO_ROOT / "src"),
    )


from scrna_benchmark.preprocessing import load_preprocessing_config


# ============================================================
# Dataset definitions
# ============================================================

DATASETS = {
    "PBMC": {
        "config": REPO_ROOT / "configs/preprocessing/pbmc.yaml",
        "role": "Primary case study",
        "technical_col": "Site",
    },

    "Lung": {
        "config": REPO_ROOT / "configs/preprocessing/lung.yaml",
        "role": "External validation",
        "technical_col": "batch",
    },

    "Kidney": {
        "config": REPO_ROOT / "configs/preprocessing/kidney.yaml",
        "role": "Ceiling control",
        "technical_col": "batch",
    },

    "Pancreas": {
        "config": REPO_ROOT / "configs/preprocessing/pancreas.yaml",
        "role": "Ceiling control",
        "technical_col": "batch",
    },

    "Blood Atlas": {
        "config": REPO_ROOT / "configs/preprocessing/blood_atlas.yaml",
        "role": "Large-cohort ceiling control",
        "technical_col": "batch",
    },
}


# ============================================================
# Formatting helpers
# ============================================================

def format_filters(filters):
    if not filters:
        return "None"

    parts = []

    for rule in filters:
        op = rule.op

        if op == "eq":
            text = f"{rule.column} = {rule.value}"

        elif op == "ne":
            text = f"{rule.column} ≠ {rule.value}"

        elif op == "notna":
            text = f"{rule.column} non-missing"

        elif op == "isna":
            text = f"{rule.column} missing"

        elif op == "in":
            text = (
                f"{rule.column} ∈ "
                f"{', '.join(map(str, rule.value))}"
            )

        elif op == "not_in":
            text = (
                f"{rule.column} excludes "
                f"{', '.join(map(str, rule.value))}"
            )

        else:
            text = (
                f"{rule.column} {rule.op} "
                f"{rule.value}"
            )

        parts.append(text)

    return "; ".join(parts)


def format_metadata_mapping(mapping):
    if not mapping:
        return "None"

    return "; ".join(
        f"{target} ← {source}"
        for target, source in mapping.items()
    )


def format_gene_filter(cfg):
    parts = []

    if cfg.exclude_var_if_true:
        parts.append(
            "exclude genes flagged in "
            + ", ".join(cfg.exclude_var_if_true)
        )

    if cfg.min_cells is not None:
        parts.append(
            f"gene detected in ≥{cfg.min_cells} cells"
        )

    return "; ".join(parts) if parts else "None"


def format_downsampling(cfg):
    if not cfg.enabled:
        return "Disabled"

    groups = " × ".join(cfg.group_cols)

    return (
        f"cap {cfg.max_cells_per_group} cells per "
        f"{groups}; seed={cfg.random_state}; "
        f"RNG={cfg.rng}"
    )


def format_expression(cfg):
    if cfg.prepare_from_counts:
        return (
            f"counts source={cfg.source}; "
            f"normalize_total({cfg.target_sum:g}) + log1p"
        )

    return (
        f"preserve supplied {cfg.source}; "
        "no renormalization/log1p"
    )


def format_hvg(cfg):
    return (
        f"{cfg.n_top_genes} genes; "
        f"batch key={cfg.batch_key}"
    )


def format_pca(cfg):
    return (
        f"compute {cfg.n_comps_compute}; "
        f"retain {cfg.n_comps_keep}"
    )


def format_harmony(cfg):
    if not cfg.enabled:
        return "Not used"

    dims = (
        cfg.n_comps
        if cfg.n_comps is not None
        else "retained PCA dimensions"
    )

    return (
        f"batch key={cfg.batch_key}; "
        f"basis={cfg.basis}; "
        f"dims={dims}"
    )


def format_scvi(cfg, representations):
    if cfg.enabled:
        epochs = (
            cfg.max_epochs
            if cfg.max_epochs is not None
            else "automatic"
        )

        return (
            f"trained on layer={cfg.layer}; "
            f"batch key={cfg.batch_key}; "
            f"latent={cfg.n_latent}; "
            f"epochs={epochs}; "
            f"early stopping={cfg.early_stopping}"
        )

    if "scvi" in representations:
        return (
            "Existing X_scVI reused; "
            "no retraining in preprocessing"
        )

    return "Unavailable / not run"


def get_n_obs(path):
    if not path.exists():
        return pd.NA

    adata = ad.read_h5ad(
        path,
        backed="r",
    )

    value = int(
        adata.n_obs
    )

    if (
        hasattr(adata, "file")
        and adata.file is not None
    ):
        adata.file.close()

    return value


# ============================================================
# Build Table S1
# ============================================================

rows = []


for dataset, spec in DATASETS.items():

    cfg = load_preprocessing_config(
        spec["config"]
    )

    input_cells = get_n_obs(
        REPO_ROOT / cfg.input_path
    )

    final_path = (
        REPO_ROOT
        / cfg.output_path
    )

    adata = ad.read_h5ad(
        final_path,
        backed="r",
    )

    final_cells = int(
        adata.n_obs
    )

    n_donors = int(
        adata.obs[
            cfg.donor_col
        ]
        .astype(str)
        .nunique()
    )

    n_celltypes = int(
        adata.obs[
            cfg.celltype_col
        ]
        .astype(str)
        .nunique()
    )

    technical_col = (
        spec[
            "technical_col"
        ]
    )

    if (
        technical_col is not None
        and technical_col
        in adata.obs.columns
    ):

        n_technical_groups = int(
            adata.obs[
                technical_col
            ]
            .astype(str)
            .nunique()
        )

    else:
        n_technical_groups = pd.NA


    if (
        hasattr(adata, "file")
        and adata.file is not None
    ):
        adata.file.close()


    reps = [
        {
            "hvg": "HVG",
            "pca": "PCA",
            "harmony": "Harmony",
            "scvi": "scVI",
        }.get(rep, rep)
        for rep
        in cfg.representations
    ]


    rows.append({
        "Dataset":
            dataset,

        "Role":
            spec["role"],

        # This is the configured preprocessing input,
        # not necessarily the raw public-source cell count.
        "Input checkpoint cells":
            input_cells,

        "Final cells":
            final_cells,

        "Donors":
            n_donors,

        "Cell types":
            n_celltypes,

        "Technical/source groups":
            n_technical_groups,

        "Biological / metadata filters":
            format_filters(
                cfg.filters
            ),

        "Metadata mapping":
            format_metadata_mapping(
                cfg.metadata_from_obs
            ),

        "Cell-type support rule":
            (
                f"≥{cfg.support.min_cells} cells; "
                f"≥{cfg.support.min_donors} donors; "
                f"{cfg.support.stage.replace('_', ' ')}"
            ),

        "Downsampling":
            format_downsampling(
                cfg.downsample
            ),

        "Gene filters":
            format_gene_filter(
                cfg.gene_filter
            ),

        "Expression source / treatment":
            format_expression(
                cfg.counts
            ),

        "HVG":
            format_hvg(
                cfg.hvg
            ),

        "PCA":
            format_pca(
                cfg.pca
            ),

        "Harmony":
            format_harmony(
                cfg.harmony
            ),

        "scVI":
            format_scvi(
                cfg.scvi,
                cfg.representations,
            ),

        "Available representations":
            ", ".join(
                reps
            ),
    })


table_s1 = pd.DataFrame(
    rows
)


display(
    table_s1
)


# Optional exports
# TABLE_DIR = REPO_ROOT / "manuscript_tables"
# TABLE_DIR.mkdir(parents=True, exist_ok=True)
#
# table_s1.to_csv(
#     TABLE_DIR / "table_S1_dataset_preprocessing_representations.csv",
#     index=False,
# )
#

,Dataset,Role,Input checkpoint cells,Final cells,Donors,Cell types,Technical/source groups,Biological / metadata filters,Metadata mapping,Cell-type support rule,Downsampling,Gene filters,Expression source / treatment,HVG,PCA,Harmony,scVI,Available representations
0,PBMC,Primary case study,62509,10107,23,13,2,Status = Healthy,None,≥200 cells; ≥5 donors; after downsampling,cap 500 cells per patient_id; seed=42; RNG=ran...,None,preserve supplied X; no renormalization/log1p,1000 genes; batch key=patient_id,compute 50; retain 15,batch key=patient_id; basis=X_pca; dims=retain...,Existing X_scVI reused; no retraining in prepr...,"HVG, PCA, Harmony, scVI"
1,Lung,External validation,584944,83966,104,31,13,disease = normal; lung_condition = Healthy; do...,cell_type ← ann_level_4; sample_id ← sample; b...,≥500 cells; ≥10 donors; before downsampling,cap 100 cells per donor_id × cell_type; seed=4...,exclude genes flagged in feature_is_filtered; ...,counts source=raw; normalize_total(10000) + log1p,1000 genes; batch key=batch,compute 50; retain 15,batch key=batch; basis=X_pca; dims=15,trained on layer=counts; batch key=batch; late...,"HVG, PCA, Harmony, scVI"
2,Kidney,Ceiling control,27675,8746,19,9,19,None,batch ← library_preparation_batch; sample_id ←...,≥200 cells; ≥5 donors; before downsampling,cap 500 cells per donor_id; seed=42; RNG=rando...,None,counts source=raw; normalize_total(10000) + log1p,1000 genes; batch key=donor_id,compute 50; retain 15,batch key=donor_id; basis=X_pca; dims=15,trained on layer=counts; batch key=donor_id; l...,"HVG, PCA, Harmony, scVI"
3,Pancreas,Ceiling control,245878,8500,17,9,5,disease = normal,batch ← Center; sample_id ← LibraryID,≥200 cells; ≥5 donors; before downsampling,cap 500 cells per donor_id; seed=42; RNG=rando...,None,counts source=raw; normalize_total(10000) + log1p,1000 genes; batch key=donor_id,compute 50; retain 15,batch key=donor_id; basis=X_pca; dims=15,trained on layer=counts; batch key=donor_id; l...,"HVG, PCA, Harmony, scVI"
4,Blood Atlas,Large-cohort ceiling control,108682,108682,166,7,14,None,None,≥200 cells; ≥5 donors; before downsampling,Disabled,None,preserve supplied X; no renormalization/log1p,1000 genes; batch key=donor_id,compute 50; retain 15,batch key=donor_id; basis=X_pca; dims=15,Unavailable / not run,"HVG, PCA, Harmony"


Table S2:

In [3]:
from pathlib import Path
import json
import sys

import pandas as pd


# ============================================================
# Locate repository
# ============================================================

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "src" / "scrna_benchmark").is_dir():
            return p

    raise RuntimeError(
        "Run this notebook from somewhere inside the repository."
    )


REPO_ROOT = find_repo_root()

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(
        0,
        str(REPO_ROOT / "src"),
    )


# ============================================================
# Read frozen primary benchmark configs
# ============================================================

PRIMARY_CONFIGS = {
    "PBMC":
        REPO_ROOT
        / "results/pbmc_no_batch/config.json",

    "Lung":
        REPO_ROOT
        / "results/lung_no_batch_L4/config.json",

    "Kidney":
        REPO_ROOT
        / "results/kidney_no_batch/config.json",

    "Pancreas":
        REPO_ROOT
        / "results/pancreas_no_batch/config.json",

    "Blood Atlas":
        REPO_ROOT
        / "results/blood_atlas_no_batch/config.json",
}


configs = {}

for dataset, path in PRIMARY_CONFIGS.items():

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        configs[
            dataset
        ] = json.load(
            handle
        )


# ============================================================
# Check common protocol values
# ============================================================

test_sizes = {
    cfg["test_size"]
    for cfg in configs.values()
}

n_folds = {
    cfg["n_folds"]
    for cfg in configs.values()
}

random_states = {
    cfg["random_state"]
    for cfg in configs.values()
}

n_random_repeats = {
    cfg["random_split_n_repeats"]
    for cfg in configs.values()
}


assert len(test_sizes) == 1
assert len(n_folds) == 1
assert len(random_states) == 1
assert len(n_random_repeats) == 1


TEST_SIZE = next(
    iter(test_sizes)
)

N_FOLDS = next(
    iter(n_folds)
)

RANDOM_STATE = next(
    iter(random_states)
)

N_RANDOM_REPEATS = next(
    iter(n_random_repeats)
)


# Default repeated-split seeds when the configs contain
# random_split_seeds = null.
RANDOM_SPLIT_SEEDS = [
    RANDOM_STATE + i
    for i in range(
        N_RANDOM_REPEATS
    )
]


# ============================================================
# Validate current implementation before documenting it
# ============================================================

models_source = (
    REPO_ROOT
    / "src/scrna_benchmark/models.py"
).read_text(
    encoding="utf-8"
)

splits_source = (
    REPO_ROOT
    / "src/scrna_benchmark/splits.py"
).read_text(
    encoding="utf-8"
)

ablation_source = (
    REPO_ROOT
    / "src/scrna_benchmark/ablation.py"
).read_text(
    encoding="utf-8"
)

mixed_source = (
    REPO_ROOT
    / "src/scrna_benchmark/mixed_models.py"
).read_text(
    encoding="utf-8"
)


assert "StandardScaler()" in models_source
assert "max_iter=5000" in models_source
assert "stratify=y" in models_source
assert "drop_first=False" in models_source

assert "np.random.default_rng(random_state)" in splits_source
assert "np.array_split(shuffled, n_folds)" in splits_source

assert "random_state + 100 * k + repeat" in ablation_source

assert "BinomialBayesMixedGLM" in mixed_source
assert 'formula = f"{outcome_col} ~ C({scheme_col})"' in mixed_source
assert '"donor_re": f"0 + C({donor_col})"' in mixed_source
assert "fit_vb()" in mixed_source


# ============================================================
# Adjustment configs
# ============================================================

with open(
    REPO_ROOT
    / "results/pbmc_with_site_covariate/config.json",
    "r",
    encoding="utf-8",
) as handle:

    pbmc_cov = json.load(
        handle
    )


with open(
    REPO_ROOT
    / "results/lung_with_batch_L4/config.json",
    "r",
    encoding="utf-8",
) as handle:

    lung_cov = json.load(
        handle
    )


# ============================================================
# Synthetic-floor config
# ============================================================

with open(
    REPO_ROOT
    / "results/synthetic_floor/config.json",
    "r",
    encoding="utf-8",
) as handle:

    synthetic_cfg = json.load(
        handle
    )


# ============================================================
# Dataset-specific donor-ablation settings
# ============================================================

ablation_details = "; ".join(
    (
        f"{dataset}: "
        f"k={cfg['donor_ablation_k_values']}, "
        f"{cfg['donor_ablation_n_repeats']} repeats"
    )
    for dataset, cfg
    in configs.items()
)


# ============================================================
# Build Table S2
# ============================================================

rows = [
    {
        "Component":
            "Classifier",

        "Setting":
            "Multinomial cell-type classification",

        "Details":
            (
                "scikit-learn LogisticRegression; "
                "max_iter=5000; random_state set by split; "
                "remaining hyperparameters use library defaults"
            ),
    },

    {
        "Component":
            "Feature scaling",

        "Setting":
            "Training-set standardization",

        "Details":
            (
                "Representation features are standardized with "
                "StandardScaler fitted on training cells only; "
                "the fitted scaler is applied to test cells"
            ),
    },

    {
        "Component":
            "Random cell-level evaluation",

        "Setting":
            (
                f"{int((1 - TEST_SIZE) * 100)}:"
                f"{int(TEST_SIZE * 100)} train:test split"
            ),

        "Details":
            (
                "Cell-level train_test_split stratified by cell type; "
                f"{N_RANDOM_REPEATS} repeats; "
                f"seeds={RANDOM_SPLIT_SEEDS}"
            ),
    },

    {
        "Component":
            "Donor-held-out evaluation",

        "Setting":
            f"{N_FOLDS}-fold donor CV",

        "Details":
            (
                "Unique donor IDs are sorted, shuffled with "
                f"np.random.default_rng({RANDOM_STATE}), and split into "
                f"{N_FOLDS} folds; donors are disjoint across train/test"
            ),
    },

    {
        "Component":
            "Class eligibility within a split",

        "Setting":
            "Train-supported labels only",

        "Details":
            (
                "Training labels require at least 2 cells; "
                "test cells whose label is absent from the training set "
                "are excluded for that donor-held-out split"
            ),
    },

    {
        "Component":
            "Primary metric",

        "Setting":
            "Macro-F1",

        "Details":
            (
                "Unweighted mean of per-class F1 scores; "
                "zero_division=0 for per-class reporting"
            ),
    },

    {
        "Component":
            "Secondary metrics",

        "Setting":
            "Accuracy and per-class diagnostics",

        "Details":
            (
                "Overall accuracy plus per-class precision, recall, "
                "F1, support, predictions, and confusion matrices"
            ),
    },

    {
        "Component":
            "Site/source covariate sensitivity",

        "Setting":
            "One-hot technical covariates appended to classifier input",

        "Details":
            (
                f"PBMC: {pbmc_cov['batch_col']}; "
                f"Lung: {lung_cov['batch_col']}. "
                "All dummy levels retained (drop_first=False); "
                "representation features are scaled separately"
            ),
    },

    {
        "Component":
            "Donor-ablation analysis",

        "Setting":
            "Vary number of training donors",

        "Details":
            (
                "k donors sampled without replacement; all remaining "
                "donors are test donors; 5 repeats per k; "
                "ablation seed = base_seed + 100*k + repeat. "
                + ablation_details
            ),
    },

    {
        "Component":
            "PBMC cross-site transfer",

        "Setting":
            "Bidirectional Cambridge ↔ Newcastle",

        "Details":
            (
                "Configured group-transfer pair: "
                f"{configs['PBMC']['group_transfer_pairs']}; "
                "classification uses the same representation scaling "
                "and logistic-regression framework"
            ),
    },

    {
        "Component":
            "Correctness GLMM",

        "Setting":
            "Separate model per representation",

        "Details":
            (
                "BinomialBayesMixedGLM: "
                "correct ~ C(evaluation scheme) + donor random intercept; "
                "donor-held-out is the reference scheme; "
                "variational-Bayes fitting (fit_vb)"
            ),
    },

    {
        "Component":
            "Synthetic floor control",

        "Setting":
            (
                f"{synthetic_cfg['n_classes']} donor-balanced "
                "random pseudo-classes"
            ),

        "Details":
            (
                f"Frozen PBMC expression/representations; "
                f"{len(synthetic_cfg['label_seeds'])} label randomizations "
                f"(seeds {synthetic_cfg['label_seeds'][0]}–"
                f"{synthetic_cfg['label_seeds'][-1]}); "
                f"random-split seed={synthetic_cfg['random_split_seeds'][0]}; "
                f"{synthetic_cfg['n_folds']} donor folds; "
                f"chance macro-F1={synthetic_cfg['chance_macro_f1']:.6f}; "
                "within each donor, pseudo-class counts differ by at most 1"
            ),
    },
]


table_s2 = pd.DataFrame(
    rows
)


display(
    table_s2
)


# Optional exports
# TABLE_DIR = REPO_ROOT / "manuscript_tables"
# TABLE_DIR.mkdir(parents=True, exist_ok=True)
#
# table_s2.to_csv(
#     TABLE_DIR / "table_S2_evaluation_protocol.csv",
#     index=False,
# )
#
# table_s2.to_latex(
#     TABLE_DIR / "table_S2_evaluation_protocol.tex",
#     index=False,
#     escape=False,
# )

,Component,Setting,Details
0,Classifier,Multinomial cell-type classification,scikit-learn LogisticRegression; max_iter=5000...
1,Feature scaling,Training-set standardization,Representation features are standardized with ...
2,Random cell-level evaluation,80:20 train:test split,Cell-level train_test_split stratified by cell...
3,Donor-held-out evaluation,5-fold donor CV,"Unique donor IDs are sorted, shuffled with np...."
4,Class eligibility within a split,Train-supported labels only,Training labels require at least 2 cells; test...
5,Primary metric,Macro-F1,Unweighted mean of per-class F1 scores; zero_d...
6,Secondary metrics,Accuracy and per-class diagnostics,"Overall accuracy plus per-class precision, rec..."
7,Site/source covariate sensitivity,One-hot technical covariates appended to class...,PBMC: Site; Lung: batch. All dummy levels reta...
8,Donor-ablation analysis,Vary number of training donors,k donors sampled without replacement; all rema...
9,PBMC cross-site transfer,Bidirectional Cambridge ↔ Newcastle,"Configured group-transfer pair: [['Cambridge',..."


Table S3:

In [4]:
from pathlib import Path

import pandas as pd


# ============================================================
# Locate repository
# ============================================================

def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()

    for p in [start, *start.parents]:
        if (p / "src" / "scrna_benchmark").is_dir():
            return p

    raise RuntimeError(
        "Run this notebook from somewhere inside the repository."
    )


REPO_ROOT = find_repo_root()


# ============================================================
# Frozen primary result directories
# ============================================================

RESULT_DIRS = {
    "PBMC":
        REPO_ROOT
        / "results/pbmc_no_batch",

    "Lung":
        REPO_ROOT
        / "results/lung_no_batch_L4",

    "Kidney":
        REPO_ROOT
        / "results/kidney_no_batch",

    "Pancreas":
        REPO_ROOT
        / "results/pancreas_no_batch",

    "Blood Atlas":
        REPO_ROOT
        / "results/blood_atlas_no_batch",
}


REP_ORDER = [
    "hvg",
    "pca",
    "harmony",
    "scvi",
]

REP_LABELS = {
    "hvg": "HVG",
    "pca": "PCA",
    "harmony": "Harmony",
    "scvi": "scVI",
}


# ============================================================
# Build complete benchmark table
# ============================================================

rows = []


for dataset, result_dir in RESULT_DIRS.items():

    random_path = (
        result_dir
        / "random_split"
        / "random_split_repeated_metrics.csv"
    )

    donor_path = (
        result_dir
        / "donor_cv"
        / "metrics.csv"
    )


    random_df = pd.read_csv(
        random_path
    )

    donor_df = pd.read_csv(
        donor_path
    )


    # Repeated random cell-level splits
    random_summary = (
        random_df
        .groupby(
            "representation",
            as_index=False,
        )
        .agg(
            random_macro_f1_mean=(
                "macro_f1",
                "mean",
            ),
            random_macro_f1_sd=(
                "macro_f1",
                "std",
            ),
            random_accuracy_mean=(
                "accuracy",
                "mean",
            ),
            random_accuracy_sd=(
                "accuracy",
                "std",
            ),
            n_random_repeats=(
                "macro_f1",
                "size",
            ),
        )
    )


    # Frozen donor-held-out summary
    donor_keep = (
        donor_df[
            [
                "representation",
                "macro_f1_mean",
                "macro_f1_std",
                "accuracy_mean",
                "accuracy_std",
                "n_folds",
            ]
        ]
        .rename(
            columns={
                "macro_f1_mean":
                    "donor_macro_f1_mean",

                "macro_f1_std":
                    "donor_macro_f1_sd",

                "accuracy_mean":
                    "donor_accuracy_mean",

                "accuracy_std":
                    "donor_accuracy_sd",
            }
        )
    )


    merged = random_summary.merge(
        donor_keep,
        on="representation",
        how="inner",
    )


    merged[
        "delta_macro_f1"
    ] = (
        merged[
            "random_macro_f1_mean"
        ]
        - merged[
            "donor_macro_f1_mean"
        ]
    )


    merged[
        "delta_accuracy"
    ] = (
        merged[
            "random_accuracy_mean"
        ]
        - merged[
            "donor_accuracy_mean"
        ]
    )


    merged[
        "Dataset"
    ] = dataset


    rows.append(
        merged
    )


table_s3_raw = pd.concat(
    rows,
    ignore_index=True,
)


# ============================================================
# Consistent ordering
# ============================================================

dataset_order = {
    name: i
    for i, name
    in enumerate(
        RESULT_DIRS
    )
}

rep_order = {
    rep: i
    for i, rep
    in enumerate(
        REP_ORDER
    )
}


table_s3_raw[
    "_dataset_order"
] = (
    table_s3_raw[
        "Dataset"
    ]
    .map(
        dataset_order
    )
)


table_s3_raw[
    "_rep_order"
] = (
    table_s3_raw[
        "representation"
    ]
    .map(
        rep_order
    )
)


table_s3_raw = (
    table_s3_raw
    .sort_values(
        [
            "_dataset_order",
            "_rep_order",
        ]
    )
    .drop(
        columns=[
            "_dataset_order",
            "_rep_order",
        ]
    )
)


# ============================================================
# Publication-facing column names
# ============================================================

table_s3 = (
    table_s3_raw
    .assign(
        Representation=lambda x:
            x[
                "representation"
            ]
            .map(
                REP_LABELS
            )
    )
    [
        [
            "Dataset",
            "Representation",

            "random_macro_f1_mean",
            "random_macro_f1_sd",

            "donor_macro_f1_mean",
            "donor_macro_f1_sd",

            "delta_macro_f1",

            "random_accuracy_mean",
            "random_accuracy_sd",

            "donor_accuracy_mean",
            "donor_accuracy_sd",

            "delta_accuracy",

            "n_random_repeats",
            "n_folds",
        ]
    ]
    .rename(
        columns={
            "random_macro_f1_mean":
                "Random macro-F1 mean",

            "random_macro_f1_sd":
                "Random macro-F1 SD",

            "donor_macro_f1_mean":
                "Donor-held-out macro-F1 mean",

            "donor_macro_f1_sd":
                "Donor-held-out macro-F1 SD",

            "delta_macro_f1":
                "Δ macro-F1",

            "random_accuracy_mean":
                "Random accuracy mean",

            "random_accuracy_sd":
                "Random accuracy SD",

            "donor_accuracy_mean":
                "Donor-held-out accuracy mean",

            "donor_accuracy_sd":
                "Donor-held-out accuracy SD",

            "delta_accuracy":
                "Δ accuracy",

            "n_random_repeats":
                "Random repeats",

            "n_folds":
                "Donor CV folds",
        }
    )
)


# ============================================================
# Display
# ============================================================

numeric_columns = [
    "Random macro-F1 mean",
    "Random macro-F1 SD",
    "Donor-held-out macro-F1 mean",
    "Donor-held-out macro-F1 SD",
    "Δ macro-F1",
    "Random accuracy mean",
    "Random accuracy SD",
    "Donor-held-out accuracy mean",
    "Donor-held-out accuracy SD",
    "Δ accuracy",
]


display(
    table_s3.style.format(
        {
            col: "{:.3f}"
            for col in numeric_columns
        }
    )
)


# Optional exports:
# keep the unrounded numeric values in the CSV.
#
# TABLE_DIR = REPO_ROOT / "manuscript_tables"
# TABLE_DIR.mkdir(parents=True, exist_ok=True)
#
# table_s3.to_csv(
#     TABLE_DIR / "table_S3_complete_primary_performance.csv",
#     index=False,
# )
#
# table_s3.to_latex(
#     TABLE_DIR / "table_S3_complete_primary_performance.tex",
#     index=False,
#     escape=False,
#     float_format="%.3f",
# )

,Dataset,Representation,Random macro-F1 mean,Random macro-F1 SD,Donor-held-out macro-F1 mean,Donor-held-out macro-F1 SD,Δ macro-F1,Random accuracy mean,Random accuracy SD,Donor-held-out accuracy mean,Donor-held-out accuracy SD,Δ accuracy,Random repeats,Donor CV folds
1,PBMC,HVG,0.796,0.007,0.768,0.014,0.028,0.825,0.008,0.806,0.018,0.019,5,5
2,PBMC,PCA,0.817,0.012,0.790,0.034,0.027,0.846,0.009,0.828,0.037,0.018,5,5
0,PBMC,Harmony,0.777,0.007,0.758,0.028,0.020,0.810,0.007,0.798,0.034,0.012,5,5
3,PBMC,scVI,0.770,0.009,0.744,0.034,0.026,0.804,0.008,0.787,0.040,0.017,5,5
5,Lung,HVG,0.880,0.003,0.859,0.007,0.021,0.858,0.002,0.839,0.005,0.019,5,5
6,Lung,PCA,0.814,0.003,0.793,0.015,0.022,0.798,0.003,0.786,0.014,0.012,5,5
4,Lung,Harmony,0.761,0.004,0.738,0.008,0.023,0.767,0.003,0.755,0.008,0.012,5,5
7,Lung,scVI,0.889,0.003,0.877,0.008,0.011,0.867,0.002,0.859,0.007,0.008,5,5
9,Kidney,HVG,0.979,0.006,0.979,0.010,-0.000,0.992,0.003,0.992,0.005,-0.001,5,5
10,Kidney,PCA,0.982,0.005,0.977,0.010,0.005,0.993,0.002,0.992,0.005,0.001,5,5
